# 00 — Phase 0: Setup, Bug Fixes & Dataset Build

Run every cell in order, top to bottom. No terminal required.

**What this notebook does**

| # | Step |
|---|------|
| 1 | Install dependencies |
| 2 | Bootstrap imports and create missing folders |
| 3 | Show the stock registry |
| 4 | Verify the two silent bugs we are fixing |
| 5 | Build every dataset from the registry |
| 6 | Smoke-test a model against proper baselines |

**Prerequisite:** the four new files must already be in `src/`:
`config.py`, `dataio.py`, `features.py`, `dataset.py`


## 1 — Install dependencies

Run once. Restart the kernel afterwards if anything was newly installed.


In [1]:
%pip install -q pandas numpy scikit-learn ta matplotlib
print('Dependencies ready. If anything installed just now, restart the kernel.')


Note: you may need to restart the kernel to use updated packages.
Dependencies ready. If anything installed just now, restart the kernel.


## 2 — Bootstrap

Adds `src/` to the import path and creates `models/`, `reports/`,
`reports/figures/` if they are missing.


In [1]:
import sys
from pathlib import Path

# Works whether the notebook is opened from notebooks/ or the project root
CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
SRC = ROOT / 'src'

assert SRC.exists(), f'Could not find src/ at {SRC}'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import config
config.ensure_dirs()

print(f'Project root : {config.PROJECT_ROOT}')
for d in config.ALL_DIRS:
    print(f'  {d.name + "/":16s} exists={d.exists()}')


Project root : C:\Users\admin\OneDrive\Documents\GitHub\Stock-Market-Predictor
  data/            exists=True
  raw/             exists=True
  processed/       exists=True
  models/          exists=True
  reports/         exists=True
  figures/         exists=True


## 3 — The stock registry

This is the whole point of Phase 0. To add a stock later you add **one entry**
to `STOCKS` in `src/config.py` and re-run this notebook. Nothing else changes.


In [2]:
import pandas as pd

rows = []
for s in config.STOCKS.values():
    rows.append({
        'key': s.key,
        'ticker': s.ticker,
        'display_name': s.display_name,
        'raw file': s.raw_filename,
        'raw exists': s.raw_path.exists(),
        'has sentiment': s.has_sentiment,
    })
pd.DataFrame(rows).set_index('key')


,ticker,display_name,raw file,raw exists,has sentiment
key,,,,,
TCS,TCS.NS,Tata Consultancy Services,TCS_raw.csv,True,True
RELIANCE,RELIANCE.NS,Reliance Industries,Reliance_raw.csv,True,False


## 4 — Verifying the bugs we are fixing

Before rebuilding anything, confirm the problems are real. This is worth
keeping in the notebook — it is direct evidence for the report.


### 4a — The fabricated final target

The original code was:

```python
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
df.dropna(inplace=True)
```

On the last row `Close.shift(-1)` is `NaN`. In Python `NaN > x` is `False`,
so `.astype(int)` turned an *unknown* label into a hard `0` (DOWN).
`dropna()` then saw no NaN and kept it. Every dataset ended with one
invented DOWN label.


In [3]:
import numpy as np

demo = pd.DataFrame({'Close': [100.0, 101.0, 103.0, 102.0]})

old_target = (demo['Close'].shift(-1) > demo['Close']).astype(int)

future = demo['Close'].shift(-1)
new_target = (future > demo['Close']).astype(float)
new_target[future.isna()] = np.nan

comparison = pd.DataFrame({
    'Close': demo['Close'],
    'Next_Close': future,
    'OLD Target': old_target,
    'NEW Target': new_target,
})
print(comparison.to_string(index=False))
print()
print('Last row: OLD invented a DOWN label. NEW marks it NaN so it is dropped.')


 Close  Next_Close  OLD Target  NEW Target
 100.0       101.0           1         1.0
 101.0       103.0           1         1.0
 103.0       102.0           0         0.0
 102.0         NaN           0         NaN

Last row: OLD invented a DOWN label. NEW marks it NaN so it is dropped.


### 4b — Row-count proof on the real file

If `data/processed/TCS_features.csv` from the old pipeline is still present,
this shows the off-by-one directly.


In [4]:
old_path = config.PROCESSED_DIR / 'TCS_features.csv'
if old_path.exists():
    old = pd.read_csv(old_path)
    print(f'Old TCS_features.csv rows : {len(old)}')
    print('Last row of the old file (Target was invented):')
    print(old.tail(1)[['Date', 'Close', 'Target']].to_string(index=False))
    print()
    print('The rebuilt dataset in step 5 should have exactly one fewer row.')
else:
    print('Old file not present - skipping this check.')


Old TCS_features.csv rows : 2416
Last row of the old file (Target was invented):
      Date       Close  Target
2023-12-29 3597.102539       0

The rebuilt dataset in step 5 should have exactly one fewer row.


### 4c — The RSI flat-window guard

The original computed `avg_gain / avg_loss` directly. Two edge cases:

| Window | Old result | New result |
|---|---|---|
| No down-days | 100 (correct by luck: `100 - 100/inf` = 100) | 100 |
| **Completely flat** | `0/0` = **NaN** -> row silently dropped | 50 (neutral) |
| No up-days | 0 | 0 |

So the old code was not as broken as it first looks - but a flat or halted
trading window produces NaN, and `dropna()` then deletes those rows without
telling you. On TCS and Reliance this never fires. On a thinly-traded stock
with halted sessions it will, and rows would vanish silently. Since the goal
is "just add new stocks", this is closed now rather than debugged later.


In [5]:
from features import add_rsi

def old_rsi(close):
    d = close.diff()
    g = d.clip(lower=0).rolling(14).mean()
    l = (-d.clip(upper=0)).rolling(14).mean()
    return 100 - (100 / (1 + g / l))

cases = {
    'strictly rising (no down days)': pd.Series(np.linspace(100, 130, 40)),
    'flat / halted stock':            pd.Series([100.0] * 40),
    'strictly falling':               pd.Series(np.linspace(130, 100, 40)),
}

rows = []
for name, s in cases.items():
    o = old_rsi(s)
    n = add_rsi(pd.DataFrame({'Close': s}))['RSI']
    rows.append({
        'case': name,
        'OLD value': o.iloc[20],
        'NEW value': n.iloc[20],
        'OLD rows lost': int(o.iloc[14:].isna().sum()),
        'NEW rows lost': int(n.iloc[14:].isna().sum()),
    })

pd.DataFrame(rows).set_index('case')


,OLD value,NEW value,OLD rows lost,NEW rows lost
case,,,,
strictly rising (no down days),100.0,100.0,0,0
flat / halted stock,NaN,50.0,26,0
strictly falling,0.0,0.0,0,0


## 5 — Build every dataset

`build_all()` walks the registry and for each stock: loads the raw CSV,
engineers all features (baseline + Ichimoku + Bollinger), attaches the target,
merges sentiment where available, audits the result, and saves it to
`data/processed/<KEY>_dataset.csv`.

Watch the audit block for `leaky_features`, `has_inf` and `duplicate_dates` —
all three must be `None`/`False`/`0`.


In [6]:
from dataset import build_all

datasets = build_all()

print()
print('=' * 50)
for k, v in datasets.items():
    print(f'{k:12s} {v.shape[0]:5d} rows x {v.shape[1]:3d} cols')



=== Building dataset: RELIANCE (Reliance Industries) ===
  loaded Reliance_raw.csv: 493 rows (1 dropped), 2022-01-03 to 2023-12-29
  features: 493 -> 443 rows (50 dropped as warm-up/unlabelled), 35 columns
  no sentiment file registered; skipping sentiment merge

Audit: RELIANCE
---------------
  rows                 : 443
  columns              : 35
  leaky_features       : None
  has_inf              : False
  nan_counts           : None
  dates_sorted         : True
  duplicate_dates      : 0
  date_range           : 2022-03-15 to 2023-12-28
  target_balance       : {1: 0.5147, 0: 0.4853}
  majority_baseline    : 0.5147
  sentiment_merged     : False
  n_features           : 20
  saved RELIANCE_dataset.csv: 443 rows x 35 cols -> processed/

=== Building dataset: TCS (Tata Consultancy Services) ===
  loaded TCS_raw.csv: 2465 rows (1 dropped), 2014-01-01 to 2023-12-29
  features: 2465 -> 2415 rows (50 dropped as warm-up/unlabelled), 35 columns
  loaded TCS_news_sentiment.csv: 1989 da

## 6 — Smoke test against honest baselines

A single 80/20 split, purely to confirm the plumbing works end to end.

**Do not read anything into these numbers.** The test set is barely a hundred
rows, so one flipped prediction moves accuracy by ~1%. Phase 1 replaces this
with walk-forward validation and reports a standard deviation. The point here
is the *comparison columns*: accuracy alone is meaningless without the
majority-class and persistence baselines beside it.


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from dataset import get_feature_columns

results = []
for key, df in datasets.items():
    feats = get_feature_columns(df)
    X, y = df[feats], df['Target']

    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, shuffle=False)

    model = RandomForestClassifier(
        n_estimators=200, random_state=config.RANDOM_STATE, n_jobs=-1
    ).fit(X_tr, y_tr)
    acc = accuracy_score(y_te, model.predict(X_te))

    majority = max(y_te.mean(), 1 - y_te.mean())

    # Persistence: predict that tomorrow repeats today's direction
    persist_pred = y_te.shift(1).fillna(y_tr.iloc[-1])
    persist = accuracy_score(y_te, persist_pred.astype(int))

    results.append({
        'stock': key,
        'rows': len(df),
        'features': len(feats),
        'RF accuracy': round(acc, 4),
        'majority': round(majority, 4),
        'persistence': round(persist, 4),
        'vs majority': round(acc - majority, 4),
    })

pd.DataFrame(results).set_index('stock')


,rows,features,RF accuracy,majority,persistence,vs majority
stock,,,,,,
RELIANCE,443,20,0.5056,0.5281,0.5169,-0.0225
TCS,565,22,0.6106,0.5398,0.5664,0.0708


## 7 — Phase 0 checklist

Confirm each of these before moving on:

- [ ] `models/`, `reports/`, `reports/figures/` now exist
- [ ] Both `<KEY>_dataset.csv` files written to `data/processed/`
- [ ] Every audit shows `leaky_features: None`
- [ ] Every audit shows `has_inf: False`
- [ ] Every audit shows `duplicate_dates: 0` and `dates_sorted: True`
- [ ] Rebuilt TCS row count is exactly one lower than the old file
- [ ] `.gitignore` edited so `data/processed/` is no longer ignored
- [ ] `bollinger.py` squeeze threshold switched to an expanding quantile

**Next:** Phase 1 — walk-forward validation harness, so the numbers in step 6
become trustworthy.
